In [ ]:
edge_list = [(0, 1), (1, 2), (0, 2), (0, 3)]


In [ ]:
import igraph

g = igraph.Graph() # Create an empty graph
g.add_vertices(4) # Add 4 vertices
g.add_edges(edge_list) # Add edges to the graph

# Plot the graph
igraph.plot(g, bbox=(150, 150), vertex_label=list(range(4)))


In [ ]:
g.get_all_simple_paths(2, to=3)


In [ ]:
g.get_shortest_paths(2, to=3)


In [ ]:
g.distances(2, 3)


In [ ]:
components = g.connected_components()


In [ ]:
print("membership: ", components.membership)  # the IDs of the component each node belongs to.
print("sizes: ", list(components.sizes()))  # the number of nodes in each component.
print("giant: ", components.giant())  # a subgraph of the largest connected component.


In [ ]:
edge_list =[(0, 1), (1, 2), (2, 1), (2, 3), (2, 5), (3, 1), (3, 4), (3, 5), (4, 5), (5, 3)]
g = igraph.Graph(directed=True)
g.add_vertices(6)
g.add_edges(edge_list)
igraph.plot(g, bbox=(250, 250), vertex_label=list(range(6)))


In [ ]:
print("From 0 to 3", g.get_all_simple_paths(0, to=3))
print("From 3 to 0", g.get_all_simple_paths(3, to=0))


In [ ]:
g.get_shortest_paths(4, 1)


In [ ]:
print(list(g.connected_components(mode="strong")))
print(list(g.connected_components(mode="weak")))


In [ ]:
# Create a graph with some triangles
edges = [(0, 1), (0, 2), (1, 2),  # Triangle: 0-1-2
         (0, 3), (3, 4), (3, 5),  # Node 3 with two neighbors (4,5)
         (4, 5),                  # Triangle: 3-4-5
         (1, 6), (6, 7)]          # Linear extension

g_cluster = igraph.Graph()
g_cluster.add_vertices(8)
g_cluster.add_edges(edges)

# Plot the graph
igraph.plot(g_cluster, bbox=(300, 200), vertex_label=list(range(8)))


In [ ]:
# Local clustering coefficient for each node
local_clustering = g_cluster.transitivity_local_undirected()

print("Local clustering coefficients:")
for i, coeff in enumerate(local_clustering):
    print(f"Node {i}: {coeff:.3f}")


In [ ]:
# Analyze clustering for specific nodes
for node in range(g_cluster.vcount()):
    neighbors = g_cluster.neighbors(node)
    degree = len(neighbors)
    clustering = local_clustering[node]

    print(f"Node {node}: degree={degree}, neighbors={neighbors}, clustering={clustering:.3f}")

    if degree >= 2:
        # Count actual triangles
        possible_edges = degree * (degree - 1) // 2
        actual_edges = 0
        for i in range(len(neighbors)):
            for j in range(i + 1, len(neighbors)):
                if g_cluster.are_adjacent(neighbors[i], neighbors[j]):
                    actual_edges += 1
        print(f"  -> {actual_edges}/{possible_edges} neighbor pairs connected")
    print()


In [ ]:
# Average local clustering (mean of local values)
avg_local_clustering = g_cluster.transitivity_avglocal_undirected()
print(f"Average local clustering: {avg_local_clustering:.3f}")

# Verify by manual calculation
import numpy as np
manual_avg = np.nanmean(local_clustering)  # nanmean ignores NaN values
print(f"Manual calculation: {manual_avg:.3f}")


In [ ]:
# Global clustering coefficient
global_clustering = g_cluster.transitivity_undirected()
print(f"Global clustering: {global_clustering:.3f}")

# Let's understand this calculation using supporting functions
# list_triangles() returns all triangles in the graph
triangles_count = len(g_cluster.list_triangles())
print(f"Number of triangles: {triangles_count}")
print(f"Triangles in graph: {g_cluster.list_triangles()}")

# Count connected triples (paths of length 2)
# degree(node) returns the degree of a specific node
triples = 0
for node in range(g_cluster.vcount()):
    degree = g_cluster.degree(node)
    # Each node with degree d contributes d*(d-1)/2 triples
    if degree >= 2:
        triples += degree * (degree - 1) // 2

print(f"Connected triples: {triples}")
print(f"Global clustering = 3 * {triangles_count} / {triples} = {3 * triangles_count / triples:.3f}")


In [ ]:
# Create different network types for comparison
import numpy as np

# 1. Complete graph (everyone connected to everyone)
n_complete = 6
g_complete = igraph.Graph.Full(n_complete)

# 2. Random graph (Erdős–Rényi)
n_random = 20
p_random = 0.2
g_random = igraph.Graph.Erdos_Renyi(n_random, p_random)

# 3. Regular ring lattice (each node connected to k nearest neighbors)
n_ring = 20
k_ring = 4
g_ring = igraph.Graph.Lattice(dim=[n_ring], circular=True, nei=k_ring//2)

networks = {
    "Complete": g_complete,
    "Random": g_random,
    "Ring Lattice": g_ring,
    "Our Example": g_cluster
}

print("Clustering Comparison:")
print("-" * 60)
print(f"{'Network':<15} {'Avg Local':<12} {'Global':<12} {'Nodes':<8} {'Edges':<8}")
print("-" * 60)

for name, graph in networks.items():
    avg_local = graph.transitivity_avglocal_undirected()
    global_clust = graph.transitivity_undirected()
    nodes = graph.vcount()
    edges = graph.ecount()

    print(f"{name:<15} {avg_local:<12.3f} {global_clust:<12.3f} {nodes:<8} {edges:<8}")


In [ ]:
# Create a small-world network (Watts-Strogatz model)
# Start with ring lattice, then rewire some edges randomly
n_ws = 30
k_ws = 6
p_rewire = 0.1

g_smallworld = igraph.Graph.Watts_Strogatz(dim=1, size=n_ws, nei=k_ws//2, p=p_rewire)

print("Small-World Network Analysis:")
print(f"Nodes: {g_smallworld.vcount()}, Edges: {g_smallworld.ecount()}")
print(f"Average local clustering: {g_smallworld.transitivity_avglocal_undirected():.3f}")
print(f"Global clustering: {g_smallworld.transitivity_undirected():.3f}")
print(f"Average path length: {g_smallworld.average_path_length():.3f}")

# Compare with random graph of same size and density
g_random_compare = igraph.Graph.Erdos_Renyi(n_ws, g_smallworld.ecount() * 2 / (n_ws * (n_ws - 1)))

print("\nCompared to random graph with same density:")
print(f"Random avg local clustering: {g_random_compare.transitivity_avglocal_undirected():.3f}")
print(f"Random global clustering: {g_random_compare.transitivity_undirected():.3f}")
print(f"Random average path length: {g_random_compare.average_path_length():.3f}")
